In [62]:
# 라이브러리 import
import getpass
import time
import pandas as pd
import plotly.graph_objects as go
import requests

from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

from tabulate import tabulate

# '강의명'을 통해, SNULIFE 강의실에서 해당 강의를 검색하고 족보 정보('이름', '연도/학기', '링크')를 반환함
def find_lecture_reference(title):
    '''
    title : 강의명 (str)
    '''

    # 1. 크롬 드라이버 경로 설정
    chromedriver_path = './chromedriver.exe'
    service = Service(executable_path=chromedriver_path)

    # 2.1. 헤드리스 옵션 설정
    chrome_options = Options()
    # chrome_options.add_argument('--headless') # 헤드리스 옵션 추가
    chrome_options.add_argument('--no-sandbox') # 안정성 장치 (1)
    chrome_options.add_argument('--disable-dev-shm-usage') # 안정성 장치 (2)
    chrome_options.add_argument('window-size=1920x1080') # headless 모드에서 창 크기 강제 설정

    # 2.2. chrome 웹 브라우저 열기
    driver = webdriver.Chrome(service=service, options=chrome_options)
    # driver.maximize_window() # 창 최대화
    
    # LOG
    print("STEP 1. Chrome 브라우저 실행 완료.")

    # 3. SNULIFE 강의실 url 접속
    driver.get("https://www.snulife.com/lecture")

    # LOG
    print("STEP 2. SNULIFE 강의실 접속 완료.")

    try:
        wait = WebDriverWait(driver, 2) # 2초 대기 설정

        # 4. 로그인
        while True:
            login_box_xpath = '//*[@id="__next"]/div/div[2]/div[2]/div[1]/div[2]/div[1]/div/form/div[1]/div'

            try:
                # 4.1. id/pw 요소 찾기
                id_box = wait.until(EC.element_to_be_clickable((By.XPATH, login_box_xpath+'/input[1]')))
                pw_box = driver.find_element(By.XPATH, login_box_xpath+'/input[2]')

                # 4.2. id/pw 요소 비우기 -> id/pw 받기 -> id/pw 입력 -> ENTER
                id_box.clear()
                id = getpass.getpass("ID를 입력해주세요.")
                id_box.send_keys(id)

                pw_box.clear()
                pw = getpass.getpass("PW를 입력해주세요.")
                pw_box.send_keys(pw) 

                pw_box.send_keys(Keys.ENTER)

                # 4.5. '로그인 실패' 팝업 확인
                wait.until(EC.visibility_of_element_located((By.XPATH, "//*[contains(text(), '아이디 또는 비밀번호를 다시 확인해주세요.')]")))

                # LOG
                print("로그인에 실패하였습니다. ID와 PW를 다시 입력해주세요.")

                # 4.6. 로그인 실패 -> '확인' 버튼 누른 후, 로그인 다시 시도
                driver.find_element(By.XPATH, '//button[text()="확인"]').click()
                
                continue
            
            # 4.7. 로그인 성공 -> 다음 로직으로 넘어가기
            except TimeoutException:
                # LOG
                print("STEP 3. 로그인 완료.")

                break

        # 5. 강의 검색
        title = title.replace(" ", "").lower() # 띄어쓰기 제거 및 소문자로 변경
        search_box = driver.find_element(By.XPATH, '//*[@id="__next"]/div/div[2]/div[2]/div[2]/div[1]/div[1]/div/form/input') # '검색 창' 찾기
        search_box.send_keys(title) # '강의명' 입력
        search_box.send_keys(Keys.ENTER)

        time.sleep(1)

        # 6.1. 검색 결과 개수 구하기
        a_xpath = '//*[@id="__next"]/div/div[2]/div/div[1]/div[3]'
        a_elem = driver.find_element(By.XPATH, a_xpath)
        a_elem_num = len(a_elem.find_elements(By.XPATH, './a'))

        # LOG
        print(f"STEP 4. 강의 검색 완료. 총 {a_elem_num}개의 강의를 찾았습니다.")

        # 6.2. 검색 결과 수집 -> list에 저장
        search_results = []

        for i in range(1, a_elem_num+1):
            tmp_xpath = f'//*[@id="__next"]/div/div[2]/div/div[1]/div[3]/a[{i}]'

            tmp_title = driver.find_element(By.XPATH, tmp_xpath+'/div[1]/div[2]').text # 강의명
            tmp_prof_name = driver.find_element(By.XPATH, tmp_xpath+'/div[2]/div[1]/span[1]').text # 교수 이름
            tmp_subject_classification = driver.find_element(By.XPATH, tmp_xpath+'/div[1]/div[1]').text # 교과 구분
            tmp_dep = driver.find_element(By.XPATH, tmp_xpath+'/div[2]/div[1]/span[3]').text # 개설 학과
            tmp_rating = driver.find_element(By.XPATH, tmp_xpath+'/div[1]/div[3]').text # 별점
            tmp_review_num = driver.find_element(By.XPATH, tmp_xpath+'/div[2]/div[2]/span[1]').text[4:] # 강의평 개수
            tmp_reference_num = driver.find_element(By.XPATH, tmp_xpath+'/div[2]/div[2]/span[3]').text[3:] # 족보 개수

            search_results.append([i, tmp_title, tmp_prof_name, tmp_subject_classification, tmp_dep, tmp_rating, tmp_review_num, tmp_reference_num])

        # 6.3. 검색 결과 반환
        if search_results:
            header_values = ['INDEX', '강의명', '교수', '교과 구분', '개설 학과', '별점', '강의평', '족보']
            # Plotly 대신 tabulate 사용 (터미널 출력용)
            print("\n" + tabulate(search_results, headers=header_values, tablefmt="grid"))
            
        else:
            print("ERROR : 조건에 맞는 강의가 존재하지 않습니다.")
            return []
        

        # 7.1. 사용자가 원하는 강의 클릭
        while True:
            try:
                target_index = int(input(f"원하는 강의의 INDEX를 입력해주세요 (1~{a_elem_num}): "))
                
                if 1 <= target_index <= a_elem_num:
                    break
                
                print("잘못된 범위입니다. 다시 입력해주세요.")

            except ValueError:
                print("숫자를 입력해주세요.")

        reference_num = int(search_results[target_index-1][-1]) # 해당 강의의 족보 개수

        # 7.2. 족보가 있는 경우
        if reference_num > 0:
            # LOG
            print(f"STEP 5. 족보 찾기 완료. 총 {reference_num}개의 족보를 찾았습니다.")

            # 7.2.1. 족보 정보 수집을 위한 페이지 이동      
            lecture_link_xpath = f'//*[@id="__next"]/div/div[2]/div/div[1]/div[3]/a[{target_index}]'
            wait.until(EC.element_to_be_clickable((By.XPATH, lecture_link_xpath))).click() # 사용자가 선택한 강의 페이지로 이동

            ref_tab_xpath = '//*[@id="__next"]/div/div[2]/div[7]/div/button[3]' 
            wait.until(EC.element_to_be_clickable((By.XPATH, ref_tab_xpath))).click() # '족보' 탭 클릭

            wait.until(lambda driver : driver.find_element(By.XPATH, '//*[@id="__next"]/div/div[2]/div[7]/div/button[3]').get_attribute('class') == 'css-1o7jdfz') # '족보' 탭이 정상적으로 클릭 완료될 때까지 대기

            # 7.2.2. 족보 정보 수집 -> list에 저장
            reference_results = []

            for i in range(1, reference_num+1): 
                base_xpath = f'//*[@id="__next"]/div/div[2]/div[8]/div/div[{i}]'
                tmp_semester = driver.find_element(By.XPATH, base_xpath+'/span').text # 연도/학기
                tmp_title = driver.find_element(By.XPATH, base_xpath+'/div[1]/span[1]').text # 파일 제목

                main_window = driver.current_window_handle # 메인 윈도우 핸들 저장
                driver.find_element(By.XPATH, base_xpath+'/div[3]/button/div').click() # '다운로드' 버튼 클릭
                
                wait.until(EC.number_of_windows_to_be(2))

                for handle in driver.window_handles:
                    if handle != driver.current_window_handle:
                        driver.switch_to.window(handle)
                        tmp_link = driver.current_url
                        driver.close()
                        break

                driver.switch_to.window(main_window)

                reference_results.append([tmp_semester, tmp_title, tmp_link])
        
            return reference_results
        
        else:
            # LOG
            print(f"ERROR : 해당 강의의 족보가 존재하지 않습니다.")

            return []

                    
    except Exception as e:
        print(f"ERROR : {e}")
        return []
    
    finally:
        driver.quit() # 크롬 드라이버 종료

In [63]:
search_query = "경영 과학 1"
ref_results = find_lecture_reference(search_query)

STEP 1. Chrome 브라우저 실행 완료.
STEP 2. SNULIFE 강의실 접속 완료.
STEP 3. 로그인 완료.
STEP 4. 강의 검색 완료. 총 6개의 강의를 찾았습니다.

+---------+------------+--------+-------------+-------------------------------+--------+----------+--------+
|   INDEX | 강의명     | 교수   | 교과 구분   | 개설 학과                     |   별점 |   강의평 |   족보 |
+=========+============+========+=============+===============================+========+==========+========+
|       1 | 경영과학 1 | 홍성필 | 전필        | 산업공학과(연합전공 기술경영) |    4.3 |        3 |      0 |
+---------+------------+--------+-------------+-------------------------------+--------+----------+--------+
|       2 | 경영과학 1 | 홍성필 | 전필        | 산업공학과(연합전공 기술경영) |    3.8 |       52 |      4 |
+---------+------------+--------+-------------+-------------------------------+--------+----------+--------+
|       3 | 경영과학 1 | 홍성필 | 전필        | 산업공학과(연합전공 기술경영) |    0   |        1 |      3 |
+---------+------------+--------+-------------+-------------------------------+--------+----------+--------+

In [168]:
# tools
'''agent가 사용할 수 있는 여러 개의 tool을 정의'''

# 1. 표준 라이브러리
import os
import time
import webbrowser

# 2. 서드 파티 라이브러리
# 2-1. LangChain 관련
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.tools import Tool
from langchain.tools.retriever import create_retriever_tool
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_google_community import GoogleSearchAPIWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 2-2. Selenium 관련
from selenium import webdriver
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# 2-3. 기타 유틸리티
from tabulate import tabulate

def search_snulife_reference(query: str):
    SNULIFE_ID = 'dlaekdna4862'
    SNULIFE_PW = 'ekdna7913'

    if not SNULIFE_ID or not SNULIFE_PW:
        return "오류: .env 파일에 SNULIFE 정보가 없습니다."

    # 입력값 파싱
    target_index, target_title, target_prof = -1, "", None

    if query.strip().isdigit():
        target_index = int(query.strip())
    
    elif "|" in query:
        parts = query.split("|")
        target_title = parts[0].strip()
        target_prof = parts[1].strip() if not parts[1].strip().isdigit() else None
        if parts[1].strip().isdigit(): target_index = int(parts[1].strip())
    
    else:
        target_title = query.strip()

    # 드라이버 설정
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--start-maximized')
    chrome_options.add_argument('--window-size=2560x1440')
    
    driver = webdriver.Chrome(service=Service('./chromedriver.exe'), options=chrome_options)
    
    try:
        driver.get("https://www.snulife.com/lecture")
        wait = WebDriverWait(driver, 5)

        # 로그인
        login_xpath = '//*[@id="__next"]/div/div[2]/div[2]/div[1]/div[2]/div[1]/div/form/div[1]/div'
        wait.until(EC.element_to_be_clickable((By.XPATH, login_xpath+'/input[1]'))).send_keys(SNULIFE_ID)
        driver.find_element(By.XPATH, login_xpath+'/input[2]').send_keys(SNULIFE_PW + Keys.ENTER)
        
        time.sleep(2)

        # 검색 (인덱스만 있는 경우는 이전 검색을 가정할 수 없으므로 에러 처리하거나 로직 보완 필요)
        if not target_title and target_index != -1:
             return "강의명을 알 수 없어 바로 선택할 수 없습니다. '강의명 | 번호'로 입력해주세요."

        search_input_xpath = '//*[@id="__next"]/div/div[2]/div[2]/div[2]/div[1]/div[1]/div/form/input'

        s_box = driver.find_element(By.XPATH, search_input_xpath)
        s_box.send_keys(target_title)
        s_box.send_keys(Keys.ENTER)

        time.sleep(2)
        
        # 강의 목록 확인
        a_xpath = '//*[@id="__next"]/div/div[2]/div/div[1]/div[3]'
        lectures = driver.find_element(By.XPATH, a_xpath).find_elements(By.XPATH, './a')
        
        final_index = -1
        lect_list = []

        # 로직: 번호 지정 > 교수명 일치 > 1개면 선택 > 아니면 목록 반환
        if target_index != -1:
            if 1 <= target_index <= len(lectures): 
                final_index = target_index
            else: 
                return "번호 범위 초과."

        else:
            found_indices = []

            for i, lect in enumerate(lectures, 1):
                try:
                    title_txt = lect.find_element(By.XPATH, './div[1]/div[2]').text
                    prof_txt = lect.find_element(By.XPATH, './div[2]/div[1]/span[1]').text
                    dept_txt = lect.find_element(By.XPATH, './div[2]/div[1]/span[3]').text
                    
                    if target_prof and target_prof == prof_txt:
                        lect_list.append(f"[{i}] {title_txt} ({prof_txt}, {dept_txt})")
                        found_indices.append(i)
                    
                    if not target_prof: 
                        lect_list.append(f"[{i}] {title_txt} ({prof_txt}, {dept_txt})")
                        found_indices.append(i)
                except: 
                    continue
            
            if len(found_indices) == 1: 
                final_index = found_indices[0]

            elif len(found_indices) == 0: 
                return "조건에 맞는 강의 없음."
            
            else: 
                return f"중복된 강의가 있습니다. 번호를 선택해주세요:\n" + "\n".join(lect_list)

        # 족보 수집
        if final_index != -1:
            # lectures[final_index-1].click()
            driver.execute_script("arguments[0].click();", lectures[final_index-1]) # (수정: 안정적)
            
            ref_btn = '//*[@id="__next"]/div/div[2]/div[7]/div/button[3]'

            # wait.until(EC.element_to_be_clickable((By.XPATH, ref_btn))).click()

            button_elem = wait.until(EC.presence_of_element_located((By.XPATH, ref_btn)))
            driver.execute_script("arguments[0].click();", button_elem)

            time.sleep(2) # 탭 전환 안정화

            # 리스트 대기
            if int(driver.find_element(By.XPATH, '//*[@id="__next"]/div/div[2]/div[7]/div/button[3]/div').text) == 0:
                return "족보 파일이 없습니다."

            items = driver.find_elements(By.XPATH, '//*[@id="__next"]/div/div[2]/div[8]/div/div')
            data = []

            # 상위 5개만 수집
            for i in range(1, min(len(items)+1, 6)):
                try:
                    base = f'//*[@id="__next"]/div/div[2]/div[8]/div/div[{i}]'
                    sem = driver.find_element(By.XPATH, base+'/span').text # 연도/학기
                    name = driver.find_element(By.XPATH, base+'/div[1]/span[1]').text # 파일 제목
                    
                    # 다운로드 링크 추출
                    main_w = driver.current_window_handle
                    driver.find_element(By.XPATH, base+'/div[3]/button/div').click()
                    wait.until(EC.number_of_windows_to_be(2))
                    
                    link = "Link Error"
                    for h in driver.window_handles:
                        if h != main_w:
                            driver.switch_to.window(h)
                            link = driver.current_url
                            driver.close()
                            break
                    driver.switch_to.window(main_w)
                    data.append([sem, name, link])
                
                except: 
                    continue
            
            return tabulate(data, headers=["학기", "제목", "링크"], tablefmt="grid")

    except Exception as e:
        return f"스크래핑 오류: {e}"
    
    finally:
        driver.quit()

In [172]:
result = search_snulife_reference("경영과학 1 | 1")

In [173]:
print(result)

족보 파일이 없습니다.
